# Individual Assignment 1 — Machine Learning Foundations  
## Data Preparation and Feature Engineering (Bank Marketing Dataset)

Name: Sacha Benayoun

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

# 1. Identifying the Prediction Target

According to the following dataset, the prediction target is the variable `y`, which indicates whether the client subscribed to a term deposit (by a yes/no output).

The variable directly represents the **business outcome** of the case studied.
Plus, it's recorded **after** the interaction which makes total sense y is what we want to predict from the given information at the time of contact.

Two variables that could superficially appear to be valid targets but should not be:

 **duration**: Call duration is only known after the call has ended, bringing out **data leakage** in the sense that at prediction time the feature won't be available, the model will use information given after output is given.
 
 **campaign** :These describe the number of contacts made during the campaign, it doesn't represent the business outcome. Furthermore, the final value is not fully known since the first call which implies unreliability of the results.

 **poutcome** : Past campaign results considered as a valid feature not a target.

# 2. Data Loading and Exploration 

In this section, we will load the dataset, focus on the structure, analyze the target variable distribution, check for missing values and visualize key features.

In [2]:
df = pd.read_csv("bank-additional.csv", sep=";")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

Shape: 4119 rows × 21 columns


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


In [3]:
#Target distribution
df["y"].value_counts()

y
no     3668
yes     451
Name: count, dtype: int64

In [4]:
#Class balance (proportions)
df["y"].value_counts(normalize=True)

y
no     0.890507
yes    0.109493
Name: proportion, dtype: float64

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             4119 non-null   int64  
 1   job             4119 non-null   object 
 2   marital         4119 non-null   object 
 3   education       4119 non-null   object 
 4   default         4119 non-null   object 
 5   housing         4119 non-null   object 
 6   loan            4119 non-null   object 
 7   contact         4119 non-null   object 
 8   month           4119 non-null   object 
 9   day_of_week     4119 non-null   object 
 10  duration        4119 non-null   int64  
 11  campaign        4119 non-null   int64  
 12  pdays           4119 non-null   int64  
 13  previous        4119 non-null   int64  
 14  poutcome        4119 non-null   object 
 15  emp.var.rate    4119 non-null   float64
 16  cons.price.idx  4119 non-null   float64
 17  cons.conf.idx   4119 non-null   f

In [6]:
# 5. Statistical summary
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,4119.0,NaN,NaN,NaN,40.11362,10.313362,18.0,32.0,38.0,47.0,88.0
job,4119,12,admin.,1012,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital,4119,4,married,2509,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education,4119,8,university.degree,1264,NaN,NaN,NaN,NaN,NaN,NaN,NaN
default,4119,3,no,3315,NaN,NaN,NaN,NaN,NaN,NaN,NaN
housing,4119,3,yes,2175,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan,4119,3,no,3349,NaN,NaN,NaN,NaN,NaN,NaN,NaN
contact,4119,2,cellular,2652,NaN,NaN,NaN,NaN,NaN,NaN,NaN
month,4119,10,may,1378,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day_of_week,4119,5,thu,860,NaN,NaN,NaN,NaN,NaN,NaN,NaN


According to following output,the dataset contains 4,119 rows and 21 columns, 10 categorical (object), 5 integer (int64), and 5 float (float64), plus the target `y`.

`y`is heavily imbalanced : 3668 "no" (89.1%) vs 451 "yes" (10.9%), so a naive classifier always predicting "no" would achieve approximately 89% accuracy without detecting any real subscriber.
For this reason, precision and recall are more appropriate metrics than accuracy

Based on the summary statistics,'pdays' is mostly 999, which means most clients were never contacted before. The column is right skewed : mean around 2.5 and median is 2.

I want also to mention that NaN appear in the statistical summary under mean, std, min and max because those statistics only apply to numerical columns, not text.


# 3. Data Splitting 

In [7]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["y"])
y = df["y"]

# First split: train (70%) and temporary (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Second split: validation (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)
print("Test size:", X_test.shape)

Train size: (2883, 20)
Validation size: (618, 20)
Test size: (618, 20)


In [8]:
print("Class distribution (%)")

print("\nFull dataset:")
print((y.value_counts(normalize=True) * 100).round(2))

print("\nTrain:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nValidation:")
print((y_val.value_counts(normalize=True) * 100).round(2))

print("\nTest:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Class distribution (%)

Full dataset:
y
no     89.05
yes    10.95
Name: proportion, dtype: float64

Train:
y
no     89.04
yes    10.96
Name: proportion, dtype: float64

Validation:
y
no     89.0
yes    11.0
Name: proportion, dtype: float64

Test:
y
no     89.16
yes    10.84
Name: proportion, dtype: float64


### Split proportions (70/15/15)

I used a 70/15/15 split:
- **Training (70%)** is used to fit the model and all preprocessing parameters.
- **Validation (15%)** is used to make modeling decisions (e.g., resampling strategy, hyperparameters).
- **Test (15%)** is kept untouched until the end to estimate final generalization performance on unseen data.

### Why stratification is necessary

The target variable `y` is imbalanced (the “yes” class is the minority).  
Stratified splitting preserves the class proportions of `y` across train/validation/test.  
Without stratification, the validation or test set could contain too few “yes” examples, making evaluation metrics (especially precision/recall) unreliable and highly variable.

### Stratification and Leakage

Stratification is necessary because the dataset is imbalanced. It ensures that train, validation, and test sets keep similar proportions of the target variable.

The split must occur before any preprocessing (imputation, scaling, SMOTE). If splitting were done later, the model would indirectly learn from validation/test data, causing data leakage and overly optimistic results.

# 4. Managing Missing Values 

In this section we inspect the dataset for missing values before any kind of splitting or preprocessing.

In [9]:
# Implicit missing values ("unknown")
for col in df.select_dtypes(include="object").columns:
    n = (df[col] == "unknown").sum()
    if n > 0:
        print(f"{col}: {n} unknown ({n/len(df)*100:.1f}%)")

job: 39 unknown (0.9%)
marital: 11 unknown (0.3%)
education: 167 unknown (4.1%)
default: 803 unknown (19.5%)
housing: 105 unknown (2.5%)
loan: 105 unknown (2.5%)


In [10]:
# Explicit missing values
df.isna().sum().sort_values(ascending=False)

age               0
campaign          0
nr.employed       0
euribor3m         0
cons.conf.idx     0
cons.price.idx    0
emp.var.rate      0
poutcome          0
previous          0
pdays             0
duration          0
job               0
day_of_week       0
month             0
contact           0
loan              0
housing           0
default           0
education         0
marital           0
y                 0
dtype: int64

In [11]:
# Sentinel numerical value
print(f"pdays = 999: {(df['pdays'] == 999).sum()} rows ({(df['pdays'] == 999).mean()*100:.1f}%)")

pdays = 999: 3959 rows (96.1%)


As indicated by the results, there is no explicit NaN values found. However, when we check for implicit missing values encoded as "unknown : default (803, ~19.5%), education 
(167, ~4%), housing and loan (105 each, ~2.5%), job (39, ~0.9%), and marital 
(11, ~0.3%). The column pdays uses 999 as a sentinel meaning "never contacted" 
(3,519 rows, ~85%).

**Strategy per vaariable** :

For job, marital, housing and loan, we will impute with the mode since missingness is low.

For default, we will create a binary indicator before imputing because 20% missingness may itself carry predictive information.

For pdays, we create a binary indicator "was_contacted_before" and replace 999 with NaN.

**Training only rule** : 

Critically, all imputation parameters (mode, median) must be fitted on the training set only. Computing these statistics on the full dataset before splitting would constitute preprocessing leakage, the model would indirectly learn from validation and test data, producing overly optimistic results

# 5. Encoding Categorical Variables

In [12]:
# Identify categorical and numerical variables (training set only)

categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical variables:", categorical_cols)
print("Numerical variables:", numerical_cols)

Categorical variables: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
Numerical variables: ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


### Encoding Categorical Variables

The dataset contains categorical variables identified from the training set.

Most categorical variables (e.g., job, marital, contact, month) are **nominal**, meaning they have no intrinsic order.

The variable **education** could be considered **ordinal** (e.g., basic → high school → university).  
However, to avoid imposing potentially incorrect linear distances between levels, One-Hot Encoding is used.

This choice allows Logistic Regression to learn separate coefficients for each category.

In [13]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

encoder = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", encoder, categorical_cols)
    ],
    remainder="passthrough"
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_val_encoded   = preprocessor.transform(X_val)
X_test_encoded  = preprocessor.transform(X_test)

print("Original feature count:", X_train.shape[1])
print("Encoded feature count:", X_train_encoded.shape[1])

Original feature count: 20
Encoded feature count: 63


The encoder was fitted using the **training set only**.

If encoding were fitted on the full dataset before splitting, category information from validation or test sets would influence training, resulting in data leakage and overly optimistic evaluation results.

### Effect of Encoding on Dimensionality

One-Hot Encoding increases dimensionality because each categorical variable is expanded into multiple binary columns (one per category).

This increases the number of model coefficients.

### Effect on Interpretability and Decision Boundaries

With One-Hot Encoding, Logistic Regression assigns one coefficient per category.

This allows the model to represent category-specific linear effects.

Without encoding, a linear model could not represent categorical distinctions properly.

However, increased dimensionality may increase the risk of overfitting if not regularized.

# 6. Feature Scaling 

In [14]:
# Identify numerical variables
numerical_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()
print("Numerical variables:", numerical_cols)

Numerical variables: ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# Fit on training only
X_train_processed = preprocessor.fit_transform(X_train)

# Transform validation and test
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed feature shape:", X_train_processed.shape)

Processed feature shape: (2883, 63)


Numerical variables were standardized using StandardScaler.

Standardization transforms features to have mean 0 and standard deviation 1.

Scaling is necessary for Logistic Regression because:
- It improves numerical stability in gradient-based optimization.
- It ensures L2 regularization penalizes all coefficients equally.
- It prevents large-scale features from dominating the loss function.
- It makes coefficients comparable across variables.

The scaler was fitted on the training set only to avoid data leakage.

### Effect on Optimization and Regularization

Logistic Regression is trained using gradient-based optimization.

If features are on different scales, the optimization landscape becomes poorly conditioned, slowing convergence and potentially causing instability.

Standardization ensures gradients are well-behaved and regularization penalties apply uniformly across features.

If scaling were performed before splitting, the mean and standard deviation would include validation/test information, causing preprocessing leakage and inflated performance estimates.

# 7. Feature Selection 

In [16]:
if "duration" in X_train.columns:
    X_train = X_train.drop(columns=["duration"])
    X_val   = X_val.drop(columns=["duration"])
    X_test  = X_test.drop(columns=["duration"])

In [17]:
import numpy as np

train_num = X_train.select_dtypes(exclude=["object"])

corr_matrix = train_num.corr().abs()

high_corr = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)

high_corr.head(10)

emp.var.rate    euribor3m         0.969648
euribor3m       nr.employed       0.942087
emp.var.rate    nr.employed       0.895815
                cons.price.idx    0.756447
cons.price.idx  euribor3m         0.654637
pdays           previous          0.607106
previous        nr.employed       0.519873
cons.price.idx  nr.employed       0.469817
previous        euribor3m         0.466142
                emp.var.rate      0.422470
dtype: float64

### Correlated features (training set only)

To detect multicollinearity, I computed the absolute Pearson correlation matrix using **only the training set numerical variables**.

I use a threshold of **|r| > 0.90** to flag *highly correlated* features. Very high correlation indicates redundant information and can make **Logistic Regression coefficients unstable**, reducing interpretability.

### Remove ONE feature from the highly correlated macroeconomic group (emp.var.rate is highly correlated with euribor3m and nr.employed)

In [18]:
to_drop_corr = ["emp.var.rate"]

X_train = X_train.drop(columns=to_drop_corr, errors="ignore")
X_val   = X_val.drop(columns=to_drop_corr, errors="ignore")
X_test  = X_test.drop(columns=to_drop_corr, errors="ignore")

print("Dropped due to high correlation:", to_drop_corr)

Dropped due to high correlation: ['emp.var.rate']


### Removal of Highly Correlated Feature

Using a threshold of |r| > 0.90, the variables `emp.var.rate` and `euribor3m` were found to be highly correlated (0.969).  

To reduce multicollinearity and improve coefficient stability in Logistic Regression, `emp.var.rate` was removed while retaining `euribor3m` and `nr.employed`.

Removing one variable from a highly correlated group reduces redundancy without significantly affecting predictive information.

In [19]:
from sklearn.feature_selection import VarianceThreshold

# Apply to numerical training data (before encoding)
train_num = X_train.select_dtypes(exclude=["object"])

vt = VarianceThreshold(threshold=0.0)
vt.fit(train_num)

print("Total numerical features:", train_num.shape[1])
print("Non-zero variance features:", vt.get_support().sum())
print("Zero variance features removed:", train_num.shape[1] - vt.get_support().sum())

Total numerical features: 8
Non-zero variance features: 8
Zero variance features removed: 0


### Low Variance Features

A threshold of 0.0 was used to identify features with zero variance.

Features with no variance contain no predictive information and cannot contribute to Logistic Regression.

No zero-variance numerical features were found (or X features were removed).

This check was performed using the training set only to avoid data leakage.

# Addressing Class Imbalance

In [20]:
print("Training set class distribution:")
print(y_train.value_counts())

print("\nTraining set class distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

Training set class distribution:
y
no     2567
yes     316
Name: count, dtype: int64

Training set class distribution (%):
y
no     89.04
yes    10.96
Name: proportion, dtype: float64


### Class Imbalance in Training Set

The training set shows a clear imbalance between the majority class ("no") and the minority class ("yes").

This imbalance is a concern because Logistic Regression may bias predictions toward the majority class, leading to misleading accuracy and poor recall for the minority class.

In [21]:
import warnings
from imblearn.over_sampling import SMOTE

# hide ONLY this harmless warning so notebook looks clean
warnings.filterwarnings("ignore", category=FutureWarning)

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_processed, y_train
)

print("Resampled class distribution:")
print(y_train_resampled.value_counts())

Resampled class distribution:
y
no     2567
yes    2567
Name: count, dtype: int64


### Resampling Strategy

SMOTE (Synthetic Minority Over-sampling Technique) was applied to the training data only.

SMOTE generates synthetic examples of the minority class by interpolating between nearby minority samples.

This helps balance the classes and allows Logistic Regression to better learn the decision boundary for the minority class.

### Why Resampling Must Occur at This Stage

Resampling is part of the training procedure and must be applied only to the training set.

If SMOTE were applied before splitting:
- Synthetic samples could appear in validation or test sets.
- This would contaminate evaluation data.
- Validation/test performance would become overly optimistic.

Therefore, SMOTE is applied after preprocessing but before model training, and only on the training set.

### Effect of Class Imbalance on Evaluation Metrics

Class imbalance affects evaluation metrics in the following ways:

- Accuracy may appear high if the model predicts only the majority class.
- Precision measures how many predicted positives are correct.
- Recall measures how many actual positives are detected.

In imbalanced datasets, recall and precision are more informative than accuracy.

SMOTE assumes that minority class samples form meaningful neighborhoods in feature space. 
By generating synthetic examples between similar minority observations, it improves the model’s ability to learn a stable decision boundary.

# Training a Logistic Regression Model

In [22]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(
    solver="saga",      
    penalty="l2",     
    C=0.01,             
    max_iter=1000,
    random_state=42
)

log_reg.fit(X_train_resampled, y_train_resampled)

LogisticRegression(C=0.01, max_iter=1000, random_state=42, solver='saga')

In [23]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred, pos_label="yes"))
print("Recall:", recall_score(y_val, y_val_pred, pos_label="yes"))

NameError: name 'y_val_pred' is not defined

In [ ]:
majority_class = y_train.value_counts().idxmax()
baseline_pred = [majority_class] * len(y_val)

print("Zero Rule Accuracy:", accuracy_score(y_val, baseline_pred))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Force correct class order
labels = ["no", "yes"]

cm = confusion_matrix(y_val, y_val_pred, labels=labels)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(values_format="d", cmap="Blues")

plt.title("Validation Confusion Matrix")
plt.show()

# Task Ordering Justification

Correct order of operations (to avoid data leakage):

1. Identify the prediction target (y).
2. Perform EDA without fitting transformations.
3. Split into train/validation/test using stratification.
4. Remove leakage features (e.g., duration) and make feature decisions using training data only.
5. Fit preprocessing (imputation, encoding, scaling) on training data only.
6. Apply SMOTE only to the training set.
7. Train the model.
8. Evaluate on validation; report final results on the test set.

Example of incorrect ordering:
Applying SMOTE or preprocessing before splitting can leak information (or synthetic samples) into validation/test data, inflating metrics.

# Task Ordering Justification

### Correct Order of Operations

1. Identify the prediction target.
2. Perform exploratory data analysis (EDA) without fitting transformations.
3. Split the dataset into training, validation, and test sets.
4. Manage missing values (fit imputers on training data only).
5. Perform feature selection using training data only.
6. Encode categorical variables (fit encoder on training data only).
7. Scale numerical features (fit scaler on training data only).
8. Apply resampling (SMOTE) to the training set only.
9. Train Logistic Regression.
10. Evaluate on validation and test sets.

#### Step-by-Step Justification

##### **Identify the prediction target**
Allowed: full dataset inspection.  
Not allowed: fitting transformations.  
Leakage risk: none at this stage.


##### **Exploratory Data Analysis**
Allowed: summary statistics, visualizations.  
Not allowed: computing preprocessing parameters (means, medians) on full dataset.  
Leakage risk: if preprocessing statistics are computed before splitting.



##### **Data Splitting**
Allowed: full dataset (X and y).  
Not allowed: performing preprocessing before splitting.  
Leakage risk: if splitting occurs later, validation/test information influences training.


##### **Managing Missing Values**
Allowed: fit imputation parameters using training data only.  
Not allowed: computing medians/modes using full dataset.  
Leakage risk: preprocessing leakage via shared statistics.



##### **Feature Selection**
Allowed: use training data only for correlation and variance checks.  
Not allowed: using validation/test data to decide which features to remove.  
Leakage risk: inflated evaluation performance if selection is based on full dataset.



##### **Encoding Categorical Variables**
Allowed: fit encoder using training data only.  
Not allowed: fitting encoder on full dataset.  
Leakage risk: category structure from validation/test influences training.



##### **Feature Scaling**
Allowed: compute mean and standard deviation using training data only.  
Not allowed: scaling before splitting.  
Leakage risk: preprocessing leakage via shared scaling parameters.



##### Addressing Class Imbalance (SMOTE)
Allowed: resample training data only.  
Not allowed: applying SMOTE before splitting or to validation/test sets.  
Leakage risk: synthetic samples contaminate validation/test, producing overly optimistic results.



##### **Model Training**
Allowed: training on processed training data only.  
Not allowed: accessing validation/test during training.



##### **Evaluation**
Allowed: evaluation on validation and test sets only after model is finalized.  
Not allowed: modifying preprocessing or model after observing test performance.


### Example of Incorrect Ordering

If SMOTE were applied before splitting, synthetic minority samples could appear in both the training and validation/test sets. This would contaminate evaluation and artificially inflate performance metrics.

Similarly, performing feature selection or scaling before splitting would allow information from validation/test data to influence model training, leading to data leakage and misleading evaluation results.